In [ ]:
###################################
# IMPORT
###################################

from polars import col, concat, lit, when
import polars as pl

import sys; sys.path.append('.'); 
import scripts.hdb_helpers as dc

In [ ]:
###################################
# DATA LOAD
###################################
hdbdata_transformed = pl.read_parquet('datalake/hdb/transformed/hdbdata')
hdbdata_transformed.glimpse()

In [ ]:
'''
REQUIREMENT: Hash this identifier column using an irreversible hashing algorithm, while preserving its
uniqueness. Explain the hashing algorithm that you adopted in your documentations.
'''

'''
We use SHA256. It is ubiquiously known to be the go to. 
SHA-256 mangles any input through 64 rounds of math into a fixed, one-way 256-bit fingerprint — you can't reverse it or fake a collision.
It's the industry default today running two decades zero known breaks.
Mathematically: The function is one-way because it's many-to-one — many possible inputs can map to the same fixed-size output, 
so there's no *unique* input to reverse back to. For any one string, you will get a humongous number of random inputs. larger than the counts of sand on earth.
'''

In [ ]:
import hashlib

In [ ]:
# # ════════════════════ SCRATCH  ══════════════════════════════════════════════════════════════════════════════════════════
# Author's note: we could do SHA256 in in polars function itself, but I would prefer to make the call to not do so with reason:
# 1) we dont want only people who know language well to be able to understand the process without looking up the terms
# 2) even if they could, there are too many nests and a polars expert would take more than seconds to confirm what is happening
# 3) list comprehension is a universal python method, much more audience can read it without looking up the terms, 
# and it has less nesting - speeding up understanding
# 4) there is no computational speed difference, nor is there even less typing comparing polars method to list comprehension
####

# hdbdata_id_hashed = (hdbdata_transformed
#                             .with_columns(
#                                 identifier_hash = col('resale_identifier')
#                                 .map_batches(
#                                     lambda s: pl.Series([hashlib.sha256(x.encode()).hexdigest() for x in s]), return_dtype=pl.String

#     ))
# )

# to_hash_list = hdbdata_transformed['resale_identifier'].to_list()
# hashed_list = [hashlib.sha256(x.encode()).hexdigest() for x in to_hash_list]

# hdbdata_id_hashed = (hdbdata_transformed
#                             .with_columns(identifier_hash = pl.Series(hashed_list))
#                             )
# # ════════════════════════════════════════════════════════════════════════════════════════════════════════

In [ ]:
############################
#### Compute Hash Lookup ###
############################
to_hash_list = hdbdata_transformed['resale_identifier'].unique().to_list()
# validate
print(to_hash_list[1:10])

In [ ]:
# Actual SHA256 HASH computation here
hash_lookup = [(x, hashlib.sha256(x.encode()).hexdigest()) for x in to_hash_list]
# validate
print(hash_lookup[1:10])

In [ ]:
df_hash_lookup = (
    pl.DataFrame(hash_lookup, schema=['resale_identifier', 'identifier_hash'], orient='row')
)
# validate
print(df_hash_lookup)

In [ ]:
############################
#### Join hash lookup to data ###
############################

In [ ]:
hdbdata_transformed_plus_hash = hdbdata_transformed.join(df_hash_lookup, on='resale_identifier', how='left')

dc.unicity(hdbdata_transformed_plus_hash, 'identifier_hash')

In [ ]:
# validate: row count should be unchanged by the join, and hash values should match the map_batches version
print(f"row count before join: {hdbdata_transformed.height:,}")
print(f"row count after join:  {hdbdata_transformed_plus_hash.height:,}")

In [ ]:
# validate
hdbdata_transformed_plus_hash.select('record_id', 'resale_identifier', 'identifier_hash').show(5)

In [ ]:
print(f"n unique identifier: {hdbdata_transformed_plus_hash['resale_identifier'].n_unique():,}")
print(f"n unique identifier_hash: {hdbdata_transformed_plus_hash['identifier_hash'].n_unique():,}")
print(f"total rows: {hdbdata_transformed_plus_hash.height:,}")

# validate look of data
dc.showall(hdbdata_transformed_plus_hash, 'closey')

In [ ]:
############################
#### Decisioning dataset ID columns to include in final data asset ###
############################
# We are here assuming that if the exploited dataset is the HASHED dataset, then we do not want the end-users to see what the unhashed value is.
# So, we make the call to replace the column [resale_dentifier] as hashed completely

# we also decide to keep the [record_id], so that it can be traced back to source data in much quicker time, should users have raise bug complaints
hdbdata_cleanly = hdbdata_transformed_plus_hash.drop('resale_identifier').rename({'identifier_hash':'resale_identifier'})

In [ ]:
#################
### DATA WRITE to Datalake
#################
OUTPUT = hdbdata_cleanly
OUTPUT.glimpse()
OUTPUT.write_parquet('datalake/hdb/hashed/hdbdata', partition_by=['tabling_version', 'month'], mkdir=True)
print('written parquet')


